In [1]:
import sys
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime
from tensorflow.keras.utils import to_categorical

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.utils.config import ExperimentConfig
from src.models.cvae import CVAE

# GPU Check
print(f"TensorFlow Version: {tf.__version__}")
print(f"Devices: {tf.config.list_physical_devices()}")

2026-01-21 09:56:54.125672: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-21 09:57:06.440180: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow Version: 2.20.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


2026-01-21 09:57:15.492971: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


## 1. Setup & Configuration

In [2]:
# Define Experiment Name
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
EXP_NAME = f"CVAE_PositionAware_{TIMESTAMP}"
OUTPUT_DIR = f"../experiments/{EXP_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize Config
# We use the Multi-Scale GAP architecture we verified
config = ExperimentConfig(
    experiment_name=f"CVAE_PositionAware_{TIMESTAMP}", # Updated Name
    input_shape=[1738, 1],
    num_classes=7,
    latent_dim=16,
    batch_size=128,
    epochs=50,
    learning_rate=0.001,
    
    # Architecture
    # --- ENCODER CONFIGURATION ---
    encoder={
        "filters": [32, 64, 128],
        "kernels": [7, 7, 5],
        "strides": [2, 2, 2],
        "use_skip_gap": False,       # Not needed for Flatten approach
        "use_global_pooling": False  # <--- FALSE = Use Flatten (Position Aware)
    },
    
    # --- DECODER CONFIGURATION ---
    decoder={
        "initial_filters": 128,
        "filters": [64, 32, 16],
        "kernels": [5, 7, 7],
        "strides": [2, 2, 2],
        "use_dense_projection": True # <--- TRUE = Use Dense Layer (Position Aware)
    },
    
    loss_weights={
        "beta": 0.5,
        "alpha": 2.0
    }
)

# Save Config for reproducibility
config.save(OUTPUT_DIR)

Config saved to ../experiments/CVAE_PositionAware_20260121_0957/config.json


## 2. Load Data

In [3]:
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_FILE = "../data/splits/manual_splits.json"

print("Loading Data...")
loader = GliomaDataLoader(DATA_PATH, split_file=SPLIT_FILE, memory_mapped=True)

# For CVAE training, we generally train on *all* training folds to maximize data utility
# But to respect the benchmark, let's train on Folds 0-3 and validate on Fold 4
fold_generator = loader.get_cross_validation_generator()

# Get the first fold (Fold 0) just to act as our primary training set for this demo
# In a full run, you might loop this or merge folds.
fold_data = next(fold_generator)
X_train, y_class_train, _ = fold_data['train']
X_val, y_class_val, _ = fold_data['val']

print(f"Training on {X_train.shape[0]} samples")
print(f"Validating on {X_val.shape[0]} samples")

Loading Data...
Loading data from ../data/processed/glioma_dataset.h5...
Data Loaded. Shape: (294710, 1738)
Unique Samples: 58
Sample ID Example: 'HF-305_v4b_1_1.h5_6' (Type: <class 'numpy.str_'>)
Loading frozen splits from ../data/splits/manual_splits.json
DEBUG: Split asks for 'HF-2544_V1B_1.h5_3'
DEBUG: Loader has 'HF-305_v4b_1_1.h5_6'
Training on 240035 samples
Validating on 54675 samples


## 3. Preprocessing for CVAE
The CVAE expects `(Spectrum, One-Hot-Label)` pairs.

In [4]:
# 1. Expand dimensions for Conv1D (N, 1738) -> (N, 1738, 1)
X_train_exp = np.expand_dims(X_train, axis=-1)
X_val_exp = np.expand_dims(X_val, axis=-1)

# 2. One-Hot Encode Labels
# We assume classes are 0..6
y_train_oh = to_categorical(y_class_train, num_classes=7)
y_val_oh = to_categorical(y_class_val, num_classes=7)

# 3. Create tf.data.Datasets (Efficient streaming)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_exp, y_train_oh))
train_dataset = train_dataset.shuffle(10000).batch(config['batch_size']).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_exp, y_val_oh))
val_dataset = val_dataset.batch(config['batch_size']).prefetch(tf.data.AUTOTUNE)

2026-01-21 09:59:37.353687: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1668723320 exceeds 10% of free system memory.


## 4. Build & Train Model

In [5]:
# Initialize Model
model = CVAE(config)

# Compile
optimizer = tf.keras.optimizers.Adam(learning_rate=config['learning_rate'])
model.compile(optimizer=optimizer)

# Call once to build graph (prints summary)
model.build(input_shape=[(None, 1738, 1), (None, 7)])
model.encoder.summary()
model.decoder.summary()

# Callbacks
callbacks = [
    # Save best weights based on validation total loss
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f"{OUTPUT_DIR}/best_weights.weights.h5",
        save_best_only=True,
        save_weights_only=True,
        monitor='val_total_loss',
        mode='min',
        verbose=1
    ),
    # Log CSV
    tf.keras.callbacks.CSVLogger(f"{OUTPUT_DIR}/training_log.csv"),
    # Early Stopping
    tf.keras.callbacks.EarlyStopping(
        monitor='val_total_loss',
        patience=10,
        restore_best_weights=True
    )
]


Model: "Encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 1738, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_conv_0 (Conv1D) │ (None, 869, 32)   │        256 │ encoder_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_bn_0            │ (None, 869, 32)   │        128 │ enc_conv_0[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_conv_1 (Conv1D) │ (None, 435, 64)   │     14,400 │ enc_bn_0[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_bn_1            │ (None, 435, 64)   │        256 │ enc_conv_1[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_conv_2 (Conv1D) │ (None, 218, 128)  │     41,088 │ enc_bn_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_bn_2            │ (None, 218, 128)  │        512 │ enc_conv_2[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_flatten     │ (None, 27904)     │          0 │ enc_bn_2[0][0]    │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 16)        │    446,480 │ spatial_flatten[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 16)        │    446,480 │ spatial_flatten[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z (Lambda)          │ (None, 16)        │          0 │ z_mean[0][0],     │
│                     │                   │            │ z_log_var[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 7)         │    195,335 │ spatial_flatten[… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,144,935 (4.37 MB)

 Trainable params: 1,144,487 (4.37 MB)

 Non-trainable params: 448 (1.75 KB)

Model: "Decoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ z_input             │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ label_input         │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 23)        │          0 │ z_input[0][0],    │
│ (Concatenate)       │                   │            │ label_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 27904)     │    669,696 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 218, 128)  │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_convT_0         │ (None, 436, 64)   │     41,024 │ reshape[0][0]     │
│ (Conv1DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_bn_0            │ (None, 436, 64)   │        256 │ dec_convT_0[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_convT_1         │ (None, 872, 32)   │     14,368 │ dec_bn_0[0][0]    │
│ (Conv1DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_bn_1            │ (None, 872, 32)   │        128 │ dec_convT_1[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_convT_2         │ (None, 1744, 16)  │      3,600 │ dec_bn_1[0][0]    │
│ (Conv1DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_bn_2            │ (None, 1744, 16)  │         64 │ dec_convT_2[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ recon_out (Conv1D)  │ (None, 1744, 1)   │        113 │ dec_bn_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cropping1d          │ (None, 1738, 1)   │          0 │ recon_out[0][0]   │
│ (Cropping1D)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 729,249 (2.78 MB)

 Trainable params: 729,025 (2.78 MB)

 Non-trainable params: 224 (896.00 B)

In [6]:

print("\nStarting Training...")
history = model.fit(
    train_dataset,
    epochs=config['epochs'],
    validation_data=val_dataset,
    callbacks=callbacks
)


Starting Training...
Epoch 1/50


2026-01-21 10:02:50.661455: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1668723320 exceeds 10% of free system memory.
2026-01-21 10:03:04.507929: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 22267392 exceeds 10% of free system memory.
2026-01-21 10:03:04.577733: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 23830016 exceeds 10% of free system memory.
2026-01-21 10:03:04.679810: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 31252480 exceeds 10% of free system memory.


 559/1876 ━━━━━━━━━━━━━━━━━━━━ 29:39 1s/step - accuracy: 0.7938 - class_loss: 0.5389 - kl_loss: 4.9208 - recon_loss: 23.6565 - total_loss: 27.1947

KeyboardInterrupt: 

## 5. Diagnostics
Check if the model learned to reconstruct.

In [ ]:
# Plot Loss Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['total_loss'], label='Train Total')
plt.plot(history.history['val_total_loss'], label='Val Total')
plt.title("Total Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['recon_loss'], label='Recon Loss')
plt.plot(history.history['val_recon_loss'], label='Val Recon')
plt.title("Reconstruction Loss")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/loss_curves.png")
plt.show()

## 6. Visual Sanity Check
Reconstruct a random validation sample.

In [ ]:
# Pick random sample
idx = np.random.randint(0, len(X_val_exp))
sample = X_val_exp[idx:idx+1]
label = y_val_oh[idx:idx+1]

# Run Inference
recon, class_pred, _, _ = model([sample, label])

plt.figure(figsize=(10, 5))
plt.plot(sample[0, :, 0], label="Original (Real)", color='black', alpha=0.7)
plt.plot(recon[0, :, 0], label="Reconstructed (CVAE)", color='red', linestyle='--')
plt.title(f"Reconstruction Check (Class {np.argmax(label)})")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/reconstruction_sample.png")
plt.show()

print(f"Predicted Class Prob: {class_pred.numpy()[0]}")
print(f"True Class: {label[0]}")